## client HTTP/REST en Python avec requests

In [ ]:
# ------------------------ utiliser les fonctions globales (get(), post(), ...) ---------------

import requests

URL = "https://gorest.co.in/public/v3/users"

# requests.(<get>,<post>,<put>,<patch>,<delete>,<head> on la même interface !!! params, data, files, headers, url, ...)
r = requests.get(URL, params={"page[size]": 5}, timeout=10, verify=False)

print(r.status_code)                    # 200
# CaseInsensitiveDict génère des clés uniques mais insensibles à la casse ==> "Content-Type" == "content-type"
print(r.headers["Content-Type"])        # application/json...
print(r.url)                            # URL réellement envoyée
print(r.elapsed.total_seconds())        # durée de l'échange

# En HTTP/REST le corpts de réponse devrait être soit application/json ou application/xml à la rigueur
if "application/json" in r.headers["Content-Type"]:
    # transforme le corps convertible en json étant donné le header , en list[dict] ou dict
    corps = r.json()
    for u in corps["data"]:
        print(u["id"], u["name"], u["status"])
    print(corps["meta"])                    # has_more, next_cursor, total

### alternatives
# si content-type == text/html ou application/xml ou text/plain ==> r.text ( str CHARSET Utf-8)
# si content-type == application/zip, video/mp4, application/octet-stream ==> r.content ( bytes )


In [ ]:
# -------------------------- RAPPEL sur getattr,  setattr, delattr, hasattr ----------

class Truc:
    pass

t = Truc()
print("attribut param ? ", hasattr(t, "param"))
setattr(t, "param", 10)
print("attribut param ? ", hasattr(t, "param"))
print(getattr(t, "param"))
delattr(t, "param")
print("attribut param ? ", hasattr(t, "param"))

print("-"*10 + "avec n'importe quoi objet comme requests" + "-"*10)
method = input("méthode http ?")
# méthode dynamique
getattr(requests, method)


In [ ]:
# ---------------------------------- idem avec la classe de base Request -----------------------
import requests
import urllib3 # dépendance tierce utilisée par requests

urllib3.disable_warnings() # enlever les warnings du verify

## REM: Request a la même signature que les fonctions globales mais rajoute la méthode

req = requests.Request(
    "GET",
    "https://gorest.co.in/public/v3/users",
    params={"filter[status]": "active", "sort": "name", "page[size]": 3},
    headers={"Accept": "application/json"},
)
prepared = req.prepare()

print(prepared.method, prepared.url)
print(dict(prepared.headers))
print(prepared.body)           # None pour un GET

##### pour envoyer la requête préparée : on doit créer un objet requests.Session(...)
s = requests.Session()
### ... envoyer la requête préparée avec .send()
r = s.send(prepared, allow_redirects=False, verify=False)

# if r.ok
if 200 <= r.status_code < 300:
    if "application/json" in r.headers["Content-Type"]:
        corps = r.json()
        for u in corps["data"]:
            print(u["id"], u["name"], u["status"])
        print(corps["meta"])
    else:
        print(f"mauvais content-type {r.headers["Content-Type"]} !")
else:
    print(f"mauvais status {r.status_code} !")



GET https://gorest.co.in/public/v3/users?filter%5Bstatus%5D=active&sort=name&page%5Bsize%5D=3
{'Accept': 'application/json'}
None
019b8c6b-4940-770f-801c-b26a9e608776 Aarav Sharma active
019c2785-b500-7297-83f7-c419f3bbaed7 Aditya Rao active
019bbaf3-3660-77aa-8329-84442f57620b Ananya Iyer active
{'has_more': True, 'next_cursor': 'eyJ2IjoiQW5hbnlhIEl5ZXIiLCJpZCI6IjAxOWJiYWYzLTM2NjAtNzdhYS04MzI5LTg0NDQyZjU3NjIwYiIsInEiOiJhMmE5YTM2MGUxOTRmNTBkIn0', 'total': None}


In [ ]:
# ------------------------ sécuriser les paramètres (injections) --------------------
from urllib.parse import quote # urllib est auto inclus avec python

# FIXES: protocole, domaine, radical, version
BASE = "https://gorest.co.in/public/v3"

# ici on fixe la ressource mais on pourrait aussi la paramétrer

def url_utilisateur(user_id: str) -> str:
    """
    retourne l'url d'un endpoint en utilisant quote qui peut échapper les paramètres de chemin ou de requêtes
    ICI: endpoint pour UN SEUL id => pas de query params => safe="" neutralise tout
    """
    return f"{BASE}/users/{quote(user_id, safe='')}"


# injection possible: /
r = requests.get(url=BASE + '/019d00aa/b240-7d95-8cc2-479f04f365d7')
print(r.url, r.json())

# / urlencodé donc neutralisé
url = url_utilisateur('019d00aa/b240-7d95-8cc2-479f04f365d7')
print(url)
r = requests.get(url=url)
print(r.url, r.json())


https://gorest.co.in/public/v3/019d00aa/b240-7d95-8cc2-479f04f365d7 {'code': 'not_found', 'detail': 'No such path in the v3 API.', 'instance': '/public/v3/019d00aa/b240-7d95-8cc2-479f04f365d7', 'request_id': '01a1e4d9-ce49-7411-a700-ce974b12d10d', 'status': 404, 'title': 'Not Found', 'type': 'https://www.gorest.co.in/docs/api/errors#not_found'}
https://gorest.co.in/public/v3/users/019d00aa%2Fb240-7d95-8cc2-479f04f365d7
https://gorest.co.in/public/v3/users/019d00aa%2Fb240-7d95-8cc2-479f04f365d7 {'code': 'invalid_parameter', 'detail': 'Parameter id is not valid: ', 'instance': '/public/v3/users/019d00aa/b240-7d95-8cc2-479f04f365d7', 'request_id': '01a1e4d9-d06a-7096-b5ca-40060a1d6646', 'status': 400, 'title': 'Bad Request', 'type': 'https://www.gorest.co.in/docs/api/errors#invalid_parameter'}
